# 00 - Train the B4 PatchTST Baseline and Export Checkpoint

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

This notebook trains and exports the **B4 PatchTST baseline**, the model configuration selected in the project's baseline comparison (see the "Baseline Selection" table in `README.md`) as the foundation for all subsequent attention-head-pruning experiments:

| seq_len | d_model | d_ff | n_heads | e_layers | pred_len |
|---:|---:|---:|---:|---:|---:|
| 336 | 128 | 256 | 8 | 3 | 96 |

B4 was chosen because it matches the forecasting accuracy of larger configurations (B1-B3) while exposing 24 total attention heads (8 heads × 3 encoder layers), giving the pruning study a reasonably sized search space.

Run this notebook whenever a fresh B4 checkpoint is needed — e.g. when the original checkpoint (trained by a teammate) is not accessible in your Google Drive. The trained checkpoint is written to:

```
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth
```

which is the exact path expected by the downstream pruning notebooks (`04_static_pruning_25_validation_test.ipynb` through `08_random_and_magnitude_baseline_pruning.ipynb`). On a T4 GPU, training for up to 10 epochs (with early stopping, patience=3) takes roughly 10-20 minutes.


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Define project paths

All experiment artifacts (checkpoints, configs, results) are stored under a shared project folder on Google Drive so that every notebook in this repository can locate them by convention.

In [2]:
from pathlib import Path
import shutil
import sys

import pandas as pd
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULT_DIR = PROJECT_DIR / "results"
CONFIG_DIR = PROJECT_DIR / "configs"

for d in [PROJECT_DIR, CHECKPOINT_DIR, RESULT_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints


## 3. Set up Time-Series-Library

PatchTST is trained through [thuml/Time-Series-Library](https://github.com/thuml/Time-Series-Library), the reference implementation used throughout this project (see notebook 01 for the original baseline sweep). The repository is cloned fresh into the Colab runtime, and a handful of optional attention-variant packages are installed as no-op dependencies so that unrelated model imports in the library do not fail.

In [3]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 33.55 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.


In [4]:
%cd /content/Time-Series-Library
!pip install -q patool sktime scikit-base --no-deps
!pip install -q --no-deps einops
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

/content/Time-Series-Library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.2 MB/s eta 0:00:00


## 4. Download the ETTh1 dataset

ETTh1 (Electricity Transformer Temperature, hourly) is the dataset used for the project's initial scope, as specified in the proposal. If a copy already exists on Drive it is reused; otherwise it is pulled from the public ETDataset repository.

In [5]:
tslib_data_dir = TSLIB_DIR / "dataset/ETDataset/ETT-small"
tslib_data_dir.mkdir(parents=True, exist_ok=True)

tslib_data_path = tslib_data_dir / "ETTh1.csv"

drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
elif not tslib_data_path.exists():
    print("Downloading ETTh1 from GitHub...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O "{tslib_data_path}"
else:
    print("ETTh1 already present.")

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (17420, 8)


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 5. Train the B4 baseline

Hyperparameters mirror the B4 row of the Baseline Selection table exactly (`seq_len=336, d_model=128, d_ff=256, n_heads=8, e_layers=3, pred_len=96, batch_size=32, train_epochs=10, patience=3, lr=1e-4`), so the resulting checkpoint is directly comparable to the numbers already reported in `README.md`. Confirm the Colab runtime is set to a T4 GPU (Runtime > Change runtime type) before running this cell.

In [6]:
%cd /content/Time-Series-Library

!python -u run.py \
  --task_name long_term_forecast \
  --is_training 1 \
  --root_path ./dataset/ETDataset/ETT-small/ \
  --data_path ETTh1.csv \
  --model_id ETTh1_336_96_dm128_h8 \
  --model PatchTST \
  --data ETTh1 \
  --features M \
  --seq_len 336 \
  --label_len 48 \
  --pred_len 96 \
  --enc_in 7 \
  --dec_in 7 \
  --c_out 7 \
  --e_layers 3 \
  --d_layers 1 \
  --factor 3 \
  --d_model 128 \
  --d_ff 256 \
  --n_heads 8 \
  --batch_size 32 \
  --train_epochs 10 \
  --patience 3 \
  --learning_rate 0.0001 \
  --des baseline_b4 \
  --itr 1

/content/Time-Series-Library
Using GPU
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           ETTh1_336_96_dm128_h8Model:              PatchTST            

Data Loader
  Data:               ETTh1               Root Path:          ./dataset/ETDataset/ETT-small/
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            336                 Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             7                   Dec In:             7                   
  C Out:              7            

## 6. Locate the checkpoint and copy it to Drive

`run.py` saves the trained checkpoint under `Time-Series-Library/checkpoints/<run-id>/checkpoint.pth`, where `<run-id>` embeds the `model_id` we passed above (`ETTh1_336_96_dm128_h8`). We locate that folder and copy it to the shared project location on Drive under the canonical name used by the rest of the pipeline, `B4_patchtst_etth1_336_dm128_h8`.

In [7]:
checkpoint_root = Path("/content/Time-Series-Library/checkpoints")

b4_matches = [
    path
    for path in checkpoint_root.iterdir()
    if "ETTh1_336_96_dm128_h8" in path.name
]

print("Found checkpoint folders:")
for path in b4_matches:
    print(path)

if len(b4_matches) != 1:
    raise RuntimeError(
        f"1 B4 checkpoint klasörü bekleniyordu, {len(b4_matches)} bulundu. "
        "Yukarıdaki listeyi kontrol et."
    )

source_b4 = b4_matches[0]
target_b4 = CHECKPOINT_DIR / "B4_patchtst_etth1_336_dm128_h8"

if target_b4.exists():
    shutil.rmtree(target_b4)

shutil.copytree(source_b4, target_b4)

print("\nB4 checkpoint Drive'a kaydedildi:")
print(target_b4)

for path in target_b4.rglob("*"):
    print(" -", path.relative_to(target_b4))

Found checkpoint folders:
/content/Time-Series-Library/checkpoints/long_term_forecast_ETTh1_336_96_dm128_h8_PatchTST_ETTh1_ftM_sl336_ll48_pl96_dm128_nh8_el3_dl1_df256_expand2_dc4_fc3_ebtimeF_dtTrue_baseline_b4_0

B4 checkpoint Drive'a kaydedildi:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8
 - checkpoint.pth


## 7. Verify the checkpoint is discoverable

The pruning notebooks locate the checkpoint with the glob pattern `B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth`. This cell runs the same lookup to confirm the exported checkpoint will be found without any path changes.

In [8]:
candidates = list(
    CHECKPOINT_DIR.glob("B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth")
)

print("Notebook 04/08'in bulacağı checkpoint(ler):")
for c in candidates:
    print(c)

assert len(candidates) == 1, "Beklenmedik sayıda checkpoint bulundu, klasör yapısını kontrol et."
print("\nHer şey hazır, artık notebook 04/08'i çalıştırabilirsin.")

Notebook 04/08'in bulacağı checkpoint(ler):
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Her şey hazır, artık notebook 04/08'i çalıştırabilirsin.
